# DPO Workflow: Style/Alignment Tuning (Qwen ~8B)

This notebook runs a **DPO (Direct Preference Optimization)** pipeline to align your **SFT Qwen ~8B** model toward your preferred *style* using **pre-built preference pairs**.

## Constraints (don’t ignore)
- **DPO requires preference pairs**: for each prompt you must have **`chosen`** and **`rejected`** responses.
- **Gemini is NOT the DPO reference model**. Gemini can be your *labeler* that produces `chosen`. The **reference model is a frozen copy of your SFT Qwen model**.
- If you include `<think>` in training targets, you are explicitly training the model to emit it. If you don’t want chain-of-thought in outputs, **do not include `<think>`** in `chosen`/`rejected`.

## Workflow Overview
1. **Config & Paths** — Define paths and run settings.
2. **Load DPO Dataset** — Expect JSONL with `prompt`, `chosen`, `rejected`.
3. **Sanity Checks** — Ensure pairs are valid, non-trivial, and consistent format.
4. **DPO Training (LoRA)** — Unsloth + TRL.
5. **Quick Eval** — Smoke test generations + optional holdout set eval.


---
## 1. Config & Paths
Set `ROOT` to the directory containing this notebook. All data and outputs live under `ROOT`.


In [ ]:
import os
import json
from pathlib import Path

ROOT = Path(os.getcwd())
if "__file__" in dir():
    ROOT = Path(__file__).resolve().parent

DATA_DIR = ROOT.parent.parent.parent / "General_dataset"
SCRIPT_DATA = ROOT / "dpo_data"
OUTPUT_DIR = ROOT / "outputs_rl_dpo"

SCRIPT_DATA.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT = {ROOT}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"SCRIPT_DATA = {SCRIPT_DATA}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")

# Paths you should set
ESG_DPO_JSONL = ROOT / "10K_dpo.jsonl"      # Final 10K balanced mix
GENERAL_DPO_JSONL = ROOT / "10K_dpo.jsonl"  # Final 10K balanced mix (Same source for split)

print(f"ESG_DPO_JSONL = {ESG_DPO_JSONL}")
print(f"GENERAL_DPO_JSONL = {GENERAL_DPO_JSONL}")


---
## 2. Install / Imports (A100)
This notebook assumes you’re on a clean GPU machine.

- If you already have a working environment from GRPO, reuse it.
- If you don’t, install: `unsloth`, `transformers`, `trl`, `datasets`, `accelerate`, `peft`.


In [ ]:
# Uncomment if needed
# !pip install -U "unsloth" "transformers" "trl" "datasets" "accelerate" "peft" "sentencepiece"

import torch
from datasets import load_dataset

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


---
## 3. DPO pairs format (you must supply)
You need a JSONL with exactly these fields:

```json
{"prompt": "...", "chosen": "...", "rejected": "..."}
```

### Non-negotiable pair rules
- `prompt` must be identical for `chosen` and `rejected`.
- `chosen` must be clearly better for the *style goal* you’re optimizing.
- `chosen` and `rejected` must be **similar length** (within ~0.5×–2×) or DPO will learn *length*, not *quality*.
- If you include `<think>...</think>`, include it in **both** chosen and rejected (same outer template) so the model learns *content* differences, not “remove `<think>`”.


In [ ]:
# Optional helper: write a small sample pairs file for debugging the pipeline.
# Replace with your real dpo_pairs.jsonl before training.

sample_pairs = [
    {
        "prompt": "Explain what additionality means in carbon credits.",
        "chosen": "<think>Define additionality, then contrast with baseline.</think>\n<answer>Additionality means...</answer>",
        "rejected": "<think>Additionality is important.</think>\n<answer>It means the project is good.</answer>",
    }
]

if not DPO_PAIRS_JSONL.exists():
    with open(DPO_PAIRS_JSONL, "w") as f:
        for row in sample_pairs:
            f.write(json.dumps(row) + "\n")

print("DPO_PAIRS_JSONL exists:", DPO_PAIRS_JSONL.exists())


---
## 4. Load dataset + sanity checks
If these checks fail, training will produce garbage.


In [ ]:
from datasets import concatenate_datasets

# 1. Load both sources
esg_ds = load_dataset("json", data_files=str(ESG_DPO_JSONL), split="train")
general_ds = load_dataset("json", data_files=str(GENERAL_DPO_JSONL), split="train")

# 2. Create high-quality holdout splits (Validation Set)
# We want ~1000 validation samples total with a 70/30 ESG/General mix.
VAL_SIZE = 1000
ESG_VAL_SIZE = int(VAL_SIZE * 0.70)  # 70%
GEN_VAL_SIZE = int(VAL_SIZE * 0.30)  # 30%

esg_split = esg_ds.train_test_split(test_size=ESG_VAL_SIZE, seed=42)
gen_split = general_ds.train_test_split(test_size=GEN_VAL_SIZE, seed=42)

# 3. Combine for Final Datasets
train_ds = concatenate_datasets([esg_split["train"], gen_split["train"]]).shuffle(seed=42)
eval_ds = concatenate_datasets([esg_split["test"], gen_split["test"]]).shuffle(seed=42)

print(f"Final Training Set: {len(train_ds)} samples")
print(f"Final Validation Set (70/30 Mix): {len(eval_ds)} samples")

# Basic schema checks
req = {"prompt", "chosen", "rejected"}
assert req.issubset(set(train_ds.column_names)), f"Missing columns: {req - set(train_ds.column_names)}"

# Spot check validation mix
for i in range(min(3, len(eval_ds))):
    ex = eval_ds[i]
    print(f"\n--- validation example {i} ---")
    print("prompt:", ex["prompt"][:150])
    print("chosen len:", len(ex["chosen"]), "rejected len:", len(ex["rejected"]))

# Quality checks
bad_same = 0
bad_len = 0
for ex in train_ds.select(range(min(len(train_ds), 2000))):
    if ex["chosen"].strip() == ex["rejected"].strip():
        bad_same += 1
    c, r = len(ex["chosen"]), len(ex["rejected"])
    if min(c, r) == 0 or max(c, r) / max(1, min(c, r)) > 2.0:
        bad_len += 1

print("bad_same (first 2k):", bad_same)
print("bad_len_ratio_gt_2x (first 2k):", bad_len)


---
## 5. DPO Training (Unsloth + TRL)
This is a **LoRA DPO** run: your SFT model is both:
- the **policy** (trainable LoRA adapters)
- the **reference** (frozen base weights)

If your goal is “nicer / more structured answers”, DPO is appropriate.
If your goal is “smarter / more correct”, DPO is the wrong tool.


In [ ]:
# If these imports fail, your env isn't set up yet.
from unsloth import FastLanguageModel
from trl import DPOTrainer
from transformers import TrainingArguments

# ---- model selection ----
# Set this to your SFT checkpoint path or HF id.
SFT_MODEL_NAME_OR_PATH = "./merged_model_400/"  # Local SFT model

max_seq_length = 8192
load_in_4bit = True  # A100 can do fp16 too; 4bit helps keep ref+policy comfortable.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_MODEL_NAME_OR_PATH,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

# Ensure reasoning tokens are handled if not already present
# Qwen3-8B should have them, but this is a safety check for the tokenizer
if "<think>" not in tokenizer.additional_special_tokens:
    tokenizer.add_special_tokens({"additional_special_tokens": ["<think>", "</think>"]})
    model.resize_token_embeddings(len(tokenizer))

# LoRA adapters (tune as needed)
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Increased for better reasoning alignment
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Loaded model + tokenizer")


In [ ]:
from transformers import TrainerCallback
import json

class JSONLCallback(TrainerCallback):
    """Custom callback to monitor ESG vs General performance separately."""
    def __init__(self, log_path, eval_dataset):
        self.log_path = log_path
        self.eval_dataset = eval_dataset
        # Pre-filter for faster evaluation
        self.esg_eval = eval_dataset.filter(lambda x: x['source_type'] == 'esg')
        self.gen_eval = eval_dataset.filter(lambda x: x['source_type'] == 'general')
        
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        trainer = kwargs.get('trainer')
        if trainer and (state.global_step % args.eval_steps == 0 or state.global_step == 0):
            print(f"\n🔍 [Custom Monitor] Step {state.global_step}: Evaluating separate datasets...")
            esg_metrics = trainer.evaluate(eval_dataset=self.esg_eval, metric_key_prefix="eval_esg")
            gen_metrics = trainer.evaluate(eval_dataset=self.gen_eval, metric_key_prefix="eval_general")
            
            # Log these to your JSONL file
            log_entry = {**esg_metrics, **gen_metrics, "step": state.global_step, "epoch": state.epoch}
            with open(self.log_path, "a") as f:
                f.write(json.dumps(log_entry) + "\n")
            
            esg_acc = esg_metrics.get('eval_esg_rewards/accuracies', 0)
            gen_acc = gen_metrics.get('eval_general_rewards/accuracies', 0)
            print(f"📊 ESG Reward Accuracy: {esg_acc:.4f}")
            print(f"📊 GENERAL Reward Accuracy: {gen_acc:.4f}")
            if gen_acc < 0.60: # Early warning threshold
                print("🚨 WARNING: General performance is dropping below 60%!")
            print("-" * 40 + "\n")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            logs["step"] = state.global_step
            logs["epoch"] = state.epoch
            with open(self.log_path, "a") as f:
                f.write(json.dumps(logs) + "\n")
            if "loss" in logs:
                print(f"Step {state.global_step}: Loss = {logs['loss']:.4f}")

# Training args
args = TrainingArguments(
    output_dir = str(OUTPUT_DIR / "dpo_lora"),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate = 5e-6,
    num_train_epochs = 1,
    logging_steps = 10,
    save_steps = 100,
    save_total_limit = 2,
    bf16 = torch.cuda.is_available(),
    fp16 = False,
    evaluation_strategy = "steps", 
    eval_steps = 100,
)

# TRL DPO expects a dataset with prompt/chosen/rejected
trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = args,
    beta = 0.05,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    tokenizer = tokenizer,
    max_length = max_seq_length,
    max_prompt_length = 1024,
    callbacks = [JSONLCallback(str(OUTPUT_DIR / "dpo_training.jsonl"), eval_ds)]
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR / "dpo_lora" / "final"))
print("Saved to", OUTPUT_DIR / "dpo_lora" / "final")


---
## 6. Quick eval / smoke test
This does not prove quality. It catches obvious failures (format collapse, refusal spikes, extreme verbosity changes).


In [ ]:
FastLanguageModel.for_inference(model)

def gen(prompt: str, max_new_tokens: int = 512):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Smoke prompts
tests = [
    "Write a concise, polite answer: What is additionality in carbon offsets?",
    "Be structured: give a short checklist for Scope 3 transport accounting.",
]

for t in tests:
    print("\n=== PROMPT ===\n", t)
    print("\n=== OUTPUT ===\n", gen(t))
